<a href="https://colab.research.google.com/github/espindolaleticia/CP02_SERS/blob/main/checkpoint_02_energia_renovavel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação — APIs de energia renovável e aprendizado de máquina

Consulte duas APIs públicas, organize os dados e resolva **duas tarefas**: classificar a fonte de empreendimentos da ANEEL e estimar a radiação solar em Petrolina (PE). Em **cada tarefa**, treine e compare **três algoritmos diferentes**. Este notebook fornece a consulta às APIs e os arquivos de dados; **você implementará a análise e os modelos**.

**Entrega:** um único link para um repositório público no GitHub com notebook executável, arquivos de dados, README e resultados. As instruções completas estão ao final e no enunciado separado.

## O que é recebido da API?

Uma requisição HTTP `GET` pede dados a uma URL. Os parâmetros identificam o recurso, o período e as variáveis. A resposta é **JSON**: um objeto com chaves e valores. Nas duas consultas desta avaliação, o acesso público **não exige login, token ou chave de API**. Se outra API exigir credenciais, não as publique no repositório.

Este código usa somente `json`, `urllib` e `csv`, bibliotecas que vêm com Python. `csv` serve apenas para exportar os dados recebidos; os imports para análise, tabelas, gráficos e ML são parte da sua implementação.

In [ ]:
import json
import urllib.parse
import urllib.request
import csv

A função a seguir monta a URL, abre a conexão, verifica o status HTTP e converte o JSON. Se houver falha na rede ou na API, exibe uma mensagem de erro.

In [ ]:
def consultar_api(endereco, parametros):
    url = endereco + "?" + urllib.parse.urlencode(parametros)
    try:
        with urllib.request.urlopen(url, timeout=30) as resposta:
            return json.load(resposta)
    except Exception as erro:
        raise RuntimeError(f"Não foi possível consultar a API: {erro}") from erro

# Tarefa 1 — Classificação (ANEEL)

**Pergunta:** a partir da potência e da localização de um empreendimento, é possível classificar sua fonte como **Solar, Eólica ou Hidráulica**?

A fonte é uma **categoria**: o problema é de classificação. O conjunto [SIGA da ANEEL](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel) registra empreendimentos em várias fases. Usamos `UFV` (solar fotovoltaica), `EOL` (eólica) e três categorias hidráulicas: `UHE`, `PCH`, `CGH`. Estas três últimas formarão uma única classe, *Hidráulica*. A tarefa trata da **categoria do cadastro**, não da quantidade de energia gerada.

### Atributos originais da resposta

| Campo da API | Significado | Uso na tarefa |
|---|---|---|
| `SigTipoGeracao` | Sigla do tipo de geração (`UFV`, `EOL`, `UHE`, `PCH`, `CGH`) | Origina o alvo `fonte`; **não entra em X** |
| `MdaPotenciaOutorgadaKw` | Potência outorgada do empreendimento, em quilowatts (kW) | Entrada `potencia_kw`; potência nominal autorizada, não energia produzida |
| `NumCoordNEmpreendimento` | Latitude aproximada do empreendimento, em graus decimais | Entrada `latitude` |
| `NumCoordEEmpreendimento` | Longitude aproximada do empreendimento, em graus decimais | Entrada `longitude` |
| `CodCEG`, `NomEmpreendimento` e campos de combustível | Código, nome e descrição associados ao empreendimento e à fonte | **Não usar como entradas**: podem revelar a classe |

A resposta CKAN contém `success` e `result`; dentro de `result`, `records` são as linhas, `fields` descrevem as colunas e `total` indica o total encontrado para o filtro. O CSV final terá **três atributos numéricos e a classe**. Quantidades consultadas têm limite por tipo e não representam a participação de cada fonte na matriz brasileira.

### 1. Consultar a ANEEL

O recurso SIGA é acessado pela API pública CKAN/DataStore. `resource_id` aponta para o conjunto; `filters` seleciona uma sigla; `limit` limita o retorno de cada pedido. Não há etapa de autenticação para esta consulta pública.

In [ ]:
URL_ANEEL = "https://dadosabertos.aneel.gov.br/api/3/action/datastore_search"
ID_RECURSO = "11ec447d-698d-4ab8-977f-b424d5deee6a"
SIGLAS = ["UFV", "EOL", "UHE", "PCH", "CGH"]
print("API ANEEL: consulta pública, sem token")

In [ ]:
registros_aneel = []
for sigla in SIGLAS:
    parametros = {
        "resource_id": ID_RECURSO,
        "filters": json.dumps({"SigTipoGeracao": sigla}),
        "limit": 1200,
    }
    resposta = consultar_api(URL_ANEEL, parametros)
    if not resposta.get("success"):
        raise ValueError(f"Consulta da ANEEL falhou para {sigla}")
    linhas = resposta["result"]["records"]
    print(f"{sigla}: {len(linhas)} linhas recebidas")
    registros_aneel.extend(linhas)
print("Total de linhas recebidas:", len(registros_aneel))

### 2. Gerar o CSV para a tarefa

A API pode fornecer números como texto, inclusive com vírgula decimal. O tratamento abaixo converte esses valores, descarta linhas sem potência ou coordenadas e transforma as siglas em três classes. Cada linha do arquivo final representa **um empreendimento**. Confira a quantidade de exemplos por classe durante sua análise; se necessário, discuta o desequilíbrio entre classes.

In [ ]:
def numero(valor):
    texto = str(valor).strip()
    if "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    try:
        return float(texto)
    except ValueError:
        return None

fontes = {"UFV": "Solar", "EOL": "Eólica", "UHE": "Hidráulica",
          "PCH": "Hidráulica", "CGH": "Hidráulica"}
linhas_classificacao = []
for registro in registros_aneel:
    potencia = numero(registro.get("MdaPotenciaOutorgadaKw"))
    latitude = numero(registro.get("NumCoordNEmpreendimento"))
    longitude = numero(registro.get("NumCoordEEmpreendimento"))
    fonte = fontes.get(registro.get("SigTipoGeracao"))
    if fonte and all(valor is not None for valor in (potencia, latitude, longitude)):
        linhas_classificacao.append([potencia, latitude, longitude, fonte])
if len(linhas_classificacao) < 90:
    raise ValueError("Poucos registros válidos: confira a resposta da ANEEL")
print("Linhas válidas:", len(linhas_classificacao))
print("Primeiro exemplo:", linhas_classificacao[0])

In [ ]:
with open("aneel_classificacao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["potencia_kw", "latitude", "longitude", "fonte"])
    escritor.writerows(linhas_classificacao)
print("Salvo: aneel_classificacao_orange.csv")

### O que você deve fazer — classificação

1. Carregue o CSV, confira os tipos das colunas, as contagens por classe, valores ausentes e distribuições relevantes. Explique `X` (três entradas) e `y` (`fonte`).
2. Faça uma divisão **estratificada** de treino/teste, por exemplo 80%/20%, com semente fixa. Ajustes de escala aprendidos dos dados devem ser feitos **somente com o treino**.
3. Escolha, justifique e treine **três classificadores diferentes** usando a mesma divisão. Eles **não estão importados nem implementados** neste notebook.
4. Compare Accuracy, Precision, Recall e F1. Declare o tipo de média usado nas métricas por classe (por exemplo, `macro`) e apresente a matriz de confusão.
5. Explique qual modelo escolheria, quais classes ele mais confunde e por que potência/localização podem não bastar para uma aplicação real. Não use `SigTipoGeracao`, nomes, CEG ou descrições da fonte como entradas.

**O que registrar:** tabela comparativa dos três algoritmos, gráfico(s) ou matriz de confusão e interpretação escrita.

## Solução — Tarefa 1 (Classificação)

A análise abaixo foi feita **sem pandas**. O CSV é carregado com `csv.DictReader`; `numpy` é usado apenas para cálculos numéricos e `scikit-learn` para os modelos.

- **X:** `potencia_kw`, `latitude`, `longitude`
- **y:** `fonte`
- `SigTipoGeracao`, nome do empreendimento, CEG e descrições da fonte não são usados como entradas, evitando vazamento do alvo.


In [ ]:
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# Leitura do CSV sem pandas
with open("aneel_classificacao_orange.csv", "r", encoding="utf-8-sig", newline="") as arquivo:
    leitor = csv.DictReader(arquivo)
    dados_classificacao = list(leitor)

print("Quantidade de registros:", len(dados_classificacao))
print("Colunas:", list(dados_classificacao[0].keys()))

# Tipos esperados após conversão
print("\nTipos usados na análise:")
print("potencia_kw -> float")
print("latitude    -> float")
print("longitude   -> float")
print("fonte       -> str")

# Valores ausentes
ausentes_classificacao = {
    coluna: sum(str(linha[coluna]).strip() == "" for linha in dados_classificacao)
    for coluna in dados_classificacao[0]
}
print("\nValores ausentes:", ausentes_classificacao)

# Contagem das classes
contagens = Counter(linha["fonte"] for linha in dados_classificacao)
print("\nContagem por classe:")
for classe, quantidade in contagens.items():
    print(f"{classe}: {quantidade}")

# Resumo numérico
colunas_numericas_class = ["potencia_kw", "latitude", "longitude"]
print("\nResumo numérico:")
for coluna in colunas_numericas_class:
    valores = np.array([float(linha[coluna]) for linha in dados_classificacao], dtype=float)
    print(
        f"{coluna:12s} | min={valores.min():.2f} | "
        f"mediana={np.median(valores):.2f} | média={valores.mean():.2f} | "
        f"máx={valores.max():.2f}"
    )


In [ ]:
# Distribuição das classes
ordem_classes = ["Solar", "Eólica", "Hidráulica"]
quantidades = [contagens.get(classe, 0) for classe in ordem_classes]

plt.figure(figsize=(7, 4))
plt.bar(ordem_classes, quantidades)
plt.title("Quantidade de empreendimentos por classe")
plt.xlabel("Fonte")
plt.ylabel("Quantidade")
plt.show()

# Distribuição da potência em escala logarítmica para reduzir o efeito dos valores extremos
potencias = np.array([float(linha["potencia_kw"]) for linha in dados_classificacao], dtype=float)

plt.figure(figsize=(7, 4))
plt.hist(np.log10(potencias + 1), bins=30)
plt.title("Distribuição da potência outorgada")
plt.xlabel("log10(potencia_kw + 1)")
plt.ylabel("Frequência")
plt.show()


In [ ]:
# X: três entradas numéricas; y: classe fonte
X_classificacao = np.array([
    [
        float(linha["potencia_kw"]),
        float(linha["latitude"]),
        float(linha["longitude"])
    ]
    for linha in dados_classificacao
], dtype=float)

y_classificacao = np.array(
    [linha["fonte"] for linha in dados_classificacao],
    dtype=object
)

# Divisão 80/20 estratificada e reprodutível
X_treino_c, X_teste_c, y_treino_c, y_teste_c = train_test_split(
    X_classificacao,
    y_classificacao,
    test_size=0.20,
    random_state=42,
    stratify=y_classificacao
)

print("Treino:", len(X_treino_c))
print("Teste :", len(X_teste_c))
print("Classes no treino:", Counter(y_treino_c))
print("Classes no teste :", Counter(y_teste_c))


### Modelos escolhidos

Foram comparados três classificadores com comportamentos diferentes:

1. **Regressão Logística:** serve como modelo linear de referência. Como é sensível à escala, foi colocada em um `Pipeline` com `StandardScaler`; o ajuste da escala ocorre somente no treino.
2. **KNN:** classifica com base nos vizinhos mais próximos. Também recebe `StandardScaler` dentro do `Pipeline`, porque distâncias seriam distorcidas se potência, latitude e longitude permanecessem em escalas muito diferentes.
3. **Random Forest:** combina várias árvores de decisão e consegue representar relações não lineares e interações entre potência e localização. Não necessita de padronização.

Todos usam exatamente a mesma divisão treino/teste.


In [ ]:
modelos_classificacao = {
    "Regressão Logística": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "KNN": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

resultados_classificacao = {}

for nome, modelo in modelos_classificacao.items():
    modelo.fit(X_treino_c, y_treino_c)
    previsoes = modelo.predict(X_teste_c)

    resultados_classificacao[nome] = {
        "accuracy": accuracy_score(y_teste_c, previsoes),
        "precision_macro": precision_score(
            y_teste_c, previsoes, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_teste_c, previsoes, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_teste_c, previsoes, average="macro", zero_division=0
        ),
        "previsoes": previsoes
    }

print("Métricas de classificação (Precision, Recall e F1 com média MACRO):")
print(f"{'Modelo':24s} {'Accuracy':>10s} {'Precision':>11s} {'Recall':>10s} {'F1':>10s}")
print("-" * 69)

for nome, r in resultados_classificacao.items():
    print(
        f"{nome:24s} "
        f"{r['accuracy']:10.4f} "
        f"{r['precision_macro']:11.4f} "
        f"{r['recall_macro']:10.4f} "
        f"{r['f1_macro']:10.4f}"
    )


In [ ]:
# Seleciona o modelo com maior F1 macro e mostra sua matriz de confusão
melhor_classificador = max(
    resultados_classificacao,
    key=lambda nome: resultados_classificacao[nome]["f1_macro"]
)

pred_melhor_c = resultados_classificacao[melhor_classificador]["previsoes"]

print("Melhor classificador por F1 macro:", melhor_classificador)

matriz = confusion_matrix(
    y_teste_c,
    pred_melhor_c,
    labels=ordem_classes
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=ordem_classes
)
disp.plot(values_format="d")
plt.title(f"Matriz de confusão — {melhor_classificador}")
plt.show()

print("Matriz numérica:")
print(matriz)


### Interpretação — classificação

Com os CSVs fornecidos, o **Random Forest** apresentou o melhor resultado: aproximadamente **97,6% de Accuracy** e **F1 macro de 0,975**. O KNN ficou muito próximo, enquanto a Regressão Logística teve desempenho menor, sugerindo que a separação entre as fontes não é puramente linear.

Na matriz de confusão do Random Forest, os erros foram poucos. A classe **Solar** foi a que apresentou mais confusões, principalmente com Hidráulica e Eólica, enquanto Hidráulica teve poucos erros.

Mesmo com métricas altas, potência e localização **não são suficientes para garantir uma classificação real perfeita**. Empreendimentos de fontes diferentes podem existir em regiões próximas e ter faixas de potência sobrepostas. Além disso, o conjunto consultado é limitado por tipo e não representa diretamente a participação das fontes na matriz elétrica brasileira. O modelo também não usa informações que revelariam diretamente a classe, como `SigTipoGeracao`, nome, CEG ou descrições de combustível.


# Tarefa 2 — Regressão (Open-Meteo)

**Pergunta:** dadas as condições meteorológicas e a hora local em Petrolina (PE), qual é a radiação solar horizontal média naquela hora? A resposta é um **número**, então o problema é de regressão.

A API histórica do [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) devolve dados horários para coordenadas e período escolhidos. Usamos **1º de abril a 30 de junho de 2025**, em horário local. São dados históricos estimados por modelos/reanálise, não leituras de um sensor particular. **Radiação em W/m² não equivale à energia em kWh nem à geração de um painel.**

### Parâmetros da requisição

| Parâmetro | Valor nesta avaliação | Significado |
|---|---|---|
| `latitude`, `longitude` | `-9.39`, `-40.50` | Coordenadas aproximadas de Petrolina (PE) |
| `start_date`, `end_date` | `2025-04-01`, `2025-06-30` | Período histórico consultado, inclusive as datas extremas |
| `timezone` | `America/Recife` | Faz `time` representar a hora local |
| `hourly` | Lista das cinco variáveis abaixo | Solicita valores horários em JSON |

### Atributos do conjunto de dados

| Campo da API → CSV | Significado e unidade | Uso na tarefa |
|---|---|---|
| `time` → `data_hora` | Data e hora local de cada registro | Ordenar e separar por tempo; não é entrada em X |
| `temperature_2m` → `temperatura_c` | Temperatura do ar a 2 m, em °C | Entrada |
| `relative_humidity_2m` → `umidade_pct` | Umidade relativa a 2 m, em % | Entrada |
| `cloud_cover` → `nuvens_pct` | Cobertura total de nuvens, em % | Entrada |
| `wind_speed_10m` → `vento_kmh` | Velocidade do vento a 10 m, em km/h | Entrada |
| `time` → `hora` | Hora local extraída da data, de 7 a 17 nesta amostra | Entrada |
| `shortwave_radiation` → `radiacao_w_m2` | Radiação solar global horizontal média da hora anterior, em W/m² | Alvo numérico `y`; **não entra em X** |

A resposta inclui `hourly` (listas dos valores, alinhadas por índice), `hourly_units` (unidades) e metadados da localização. Cada linha do CSV final representa **uma hora entre 7h e 17h**; esse recorte reduz o domínio de horas noturnas, mas não remove a importância da hora para a previsão.

### 3. Consultar o histórico meteorológico

A chamada usa GET, sem cadastro ou chave para esta consulta pública de uso educacional. Se `hourly` faltar, verifique datas e parâmetros.

In [ ]:
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"
parametros_meteo = {
    "latitude": -9.39,
    "longitude": -40.50,
    "start_date": "2025-04-01",
    "end_date": "2025-06-30",
    "timezone": "America/Recife",
    "hourly": ",".join(["temperature_2m", "relative_humidity_2m",
                         "cloud_cover", "wind_speed_10m", "shortwave_radiation"]),
}
print("Open-Meteo: consulta pública, sem token")

In [ ]:
resposta_meteo = consultar_api(URL_METEO, parametros_meteo)
if "hourly" not in resposta_meteo:
    raise ValueError("Resposta sem a seção hourly")
horas_api = resposta_meteo["hourly"]
print("Campos recebidos:", list(horas_api))
print("Unidades informadas pela API:", resposta_meteo.get("hourly_units", {}))
print("Quantidade de horários:", len(horas_api["time"]))

### 4. Gerar o CSV horário

As listas de `hourly` são alinhadas: o índice 0 de cada variável se refere ao índice 0 de `time`. Percorremos esses índices, mantemos as horas de 7h a 17h e descartamos linhas com valores ausentes. A data permanece ordenada para facilitar uma avaliação temporal.

In [ ]:
nomes_api = ["temperature_2m", "relative_humidity_2m", "cloud_cover",
             "wind_speed_10m", "shortwave_radiation"]
linhas_regressao = []
for indice, data_hora in enumerate(horas_api["time"]):
    hora = int(data_hora[11:13])
    valores = [horas_api[nome][indice] for nome in nomes_api]
    if 7 <= hora <= 17 and all(valor is not None for valor in valores):
        linhas_regressao.append([data_hora, *valores[:4], hora, valores[4]])
if len(linhas_regressao) < 100:
    raise ValueError("Poucos horários válidos: confira a resposta do Open-Meteo")
print("Horas válidas:", len(linhas_regressao))
print("Primeiro exemplo:", linhas_regressao[0])

In [ ]:
with open("meteo_regressao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["data_hora", "temperatura_c", "umidade_pct",
                       "nuvens_pct", "vento_kmh", "hora", "radiacao_w_m2"])
    escritor.writerows(linhas_regressao)
print("Salvo: meteo_regressao_orange.csv")

### O que você deve fazer — regressão

1. Carregue o CSV, explique as colunas, examine valores ausentes e apresente ao menos uma visualização. Defina `X` (cinco entradas) e `y` (`radiacao_w_m2`).
2. Mantenha as linhas em ordem de `data_hora`: use aproximadamente **80% das primeiras horas para treino** e as 20% finais para teste. Evite embaralhar as horas. Ajuste transformações apenas com o treino.
3. Escolha, justifique e treine **três algoritmos de regressão diferentes** com a mesma divisão. Os imports e a implementação são sua responsabilidade.
4. Compare **MAE** (W/m²), **MSE** ((W/m²)²) e **R²**; faça um gráfico de valores reais × previstos de pelo menos um modelo.
5. Interprete os erros, discuta o peso da hora e explique por que essa estimativa da radiação **não prevê automaticamente a energia produzida** por um sistema fotovoltaico.

**O que registrar:** tabela comparativa dos três algoritmos, gráfico e interpretação escrita. Nenhum valor de `radiacao_w_m2` deve ser usado para criar uma entrada do mesmo registro.

## Solução — Tarefa 2 (Regressão)

A regressão também é feita **sem pandas**. As linhas são mantidas em ordem de `data_hora`.

- **X:** `temperatura_c`, `umidade_pct`, `nuvens_pct`, `vento_kmh`, `hora`
- **y:** `radiacao_w_m2`
- `data_hora` é usada somente para ordenar e separar os períodos; não entra como variável numérica.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

with open("meteo_regressao_orange.csv", "r", encoding="utf-8-sig", newline="") as arquivo:
    leitor = csv.DictReader(arquivo)
    dados_regressao = list(leitor)

# Garante explicitamente a ordem temporal
dados_regressao.sort(key=lambda linha: linha["data_hora"])

print("Quantidade de registros:", len(dados_regressao))
print("Colunas:", list(dados_regressao[0].keys()))

print("\nTipos usados na análise:")
print("data_hora       -> str (ordenação temporal)")
print("temperatura_c   -> float")
print("umidade_pct     -> float")
print("nuvens_pct      -> float")
print("vento_kmh       -> float")
print("hora            -> float")
print("radiacao_w_m2   -> float")

ausentes_regressao = {
    coluna: sum(str(linha[coluna]).strip() == "" for linha in dados_regressao)
    for coluna in dados_regressao[0]
}
print("\nValores ausentes:", ausentes_regressao)

colunas_numericas_reg = [
    "temperatura_c", "umidade_pct", "nuvens_pct",
    "vento_kmh", "hora", "radiacao_w_m2"
]

print("\nResumo numérico:")
for coluna in colunas_numericas_reg:
    valores = np.array([float(linha[coluna]) for linha in dados_regressao], dtype=float)
    print(
        f"{coluna:15s} | min={valores.min():.2f} | "
        f"mediana={np.median(valores):.2f} | média={valores.mean():.2f} | "
        f"máx={valores.max():.2f}"
    )


In [ ]:
# Visualização: radiação média observada por hora do dia
radiacao_por_hora = {}

for linha in dados_regressao:
    hora = int(float(linha["hora"]))
    radiacao_por_hora.setdefault(hora, []).append(float(linha["radiacao_w_m2"]))

horas = sorted(radiacao_por_hora)
medias_radiacao = [np.mean(radiacao_por_hora[h]) for h in horas]

plt.figure(figsize=(8, 4))
plt.plot(horas, medias_radiacao, marker="o")
plt.title("Radiação solar média por hora local")
plt.xlabel("Hora")
plt.ylabel("Radiação média (W/m²)")
plt.xticks(horas)
plt.grid(alpha=0.3)
plt.show()

for h, media in zip(horas, medias_radiacao):
    print(f"{h:02d}h: {media:.1f} W/m²")


In [ ]:
X_regressao = np.array([
    [
        float(linha["temperatura_c"]),
        float(linha["umidade_pct"]),
        float(linha["nuvens_pct"]),
        float(linha["vento_kmh"]),
        float(linha["hora"])
    ]
    for linha in dados_regressao
], dtype=float)

y_regressao = np.array(
    [float(linha["radiacao_w_m2"]) for linha in dados_regressao],
    dtype=float
)

datas_regressao = np.array(
    [linha["data_hora"] for linha in dados_regressao],
    dtype=object
)

# Primeiras ~80% das horas para treino; últimas ~20% para teste.
# Não há embaralhamento.
corte = int(len(X_regressao) * 0.80)

X_treino_r = X_regressao[:corte]
X_teste_r = X_regressao[corte:]
y_treino_r = y_regressao[:corte]
y_teste_r = y_regressao[corte:]

datas_treino_r = datas_regressao[:corte]
datas_teste_r = datas_regressao[corte:]

print("Treino:", len(X_treino_r), "registros")
print("Teste :", len(X_teste_r), "registros")
print("Período de treino:", datas_treino_r[0], "até", datas_treino_r[-1])
print("Período de teste :", datas_teste_r[0], "até", datas_teste_r[-1])


### Modelos escolhidos

1. **Regressão Linear:** referência simples para verificar o quanto uma relação aproximadamente linear explica a radiação. A escala é ajustada apenas com os dados de treino por meio de `Pipeline`.
2. **KNN Regressor:** prevê a partir de condições meteorológicas parecidas. Como depende de distância, usa `StandardScaler` dentro do `Pipeline`.
3. **Random Forest Regressor:** captura relações não lineares, especialmente a curva da radiação ao longo do dia e interações entre variáveis meteorológicas.

Os três usam exatamente o mesmo corte temporal.


In [ ]:
modelos_regressao = {
    "Regressão Linear": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", LinearRegression())
    ]),
    "KNN Regressor": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", KNeighborsRegressor(n_neighbors=7, weights="distance"))
    ]),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    )
}

resultados_regressao = {}

for nome, modelo in modelos_regressao.items():
    modelo.fit(X_treino_r, y_treino_r)
    previsoes = modelo.predict(X_teste_r)

    resultados_regressao[nome] = {
        "mae": mean_absolute_error(y_teste_r, previsoes),
        "mse": mean_squared_error(y_teste_r, previsoes),
        "r2": r2_score(y_teste_r, previsoes),
        "previsoes": previsoes
    }

print("Métricas de regressão:")
print(f"{'Modelo':25s} {'MAE (W/m²)':>12s} {'MSE ((W/m²)²)':>16s} {'R²':>10s}")
print("-" * 68)

for nome, r in resultados_regressao.items():
    print(
        f"{nome:25s} "
        f"{r['mae']:12.2f} "
        f"{r['mse']:16.2f} "
        f"{r['r2']:10.4f}"
    )


In [ ]:
# Modelo escolhido para visualização: menor MAE
melhor_regressor = min(
    resultados_regressao,
    key=lambda nome: resultados_regressao[nome]["mae"]
)

pred_melhor_r = resultados_regressao[melhor_regressor]["previsoes"]

print("Melhor regressor por MAE:", melhor_regressor)

plt.figure(figsize=(6, 6))
plt.scatter(y_teste_r, pred_melhor_r, alpha=0.65)

limite_min = min(y_teste_r.min(), pred_melhor_r.min())
limite_max = max(y_teste_r.max(), pred_melhor_r.max())
plt.plot([limite_min, limite_max], [limite_min, limite_max], linestyle="--")

plt.title(f"Valores reais × previstos — {melhor_regressor}")
plt.xlabel("Radiação real (W/m²)")
plt.ylabel("Radiação prevista (W/m²)")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Importância das variáveis no Random Forest
rf_regressor = modelos_regressao["Random Forest Regressor"]
nomes_entradas = [
    "temperatura_c", "umidade_pct", "nuvens_pct", "vento_kmh", "hora"
]

print("Importância das variáveis no Random Forest:")
for nome, importancia in sorted(
    zip(nomes_entradas, rf_regressor.feature_importances_),
    key=lambda par: par[1],
    reverse=True
):
    print(f"{nome:15s}: {importancia:.4f}")


### Interpretação — regressão

Com os arquivos fornecidos, **Random Forest Regressor** e **KNN Regressor** tiveram resultados muito próximos. O Random Forest obteve o menor MAE, cerca de **67,2 W/m²**, e R² de aproximadamente **0,842**. O KNN apresentou MAE de cerca de **67,7 W/m²** e R² também próximo de **0,842**. A Regressão Linear ficou bem abaixo, com MAE em torno de **145 W/m²** e R² de aproximadamente **0,36**, mostrando que a relação não é bem descrita por uma reta simples.

A **hora** tem grande peso porque a radiação segue o ciclo diário: no arquivo analisado, a média cresce desde 7h, fica mais alta perto do meio-dia e cai no fim da tarde. No Random Forest, `hora` aparece como a variável de maior importância, em torno de **0,49**. Essa importância é uma medida interna do modelo e não deve ser interpretada como causalidade.

Os erros podem ocorrer porque a radiação também depende de condições atmosféricas e porque as variáveis disponíveis não descrevem todos os fatores físicos relevantes. Além disso, prever `radiacao_w_m2` **não equivale a prever automaticamente a energia gerada por um sistema fotovoltaico**. Para estimar geração seriam necessárias informações do sistema, como área e eficiência dos painéis, inclinação/orientação, potência instalada e perdas.


# Conclusão geral

As duas tarefas foram resolvidas sem pandas e sem usar atributos que vazam diretamente os alvos.

- **Classificação:** o melhor resultado nos dados fornecidos foi obtido pelo Random Forest, com F1 macro próximo de 0,975.
- **Regressão:** Random Forest e KNN tiveram desempenho semelhante; o Random Forest apresentou o menor MAE, próximo de 67 W/m², com R² de aproximadamente 0,842.
- As divisões respeitaram o enunciado: classificação estratificada e regressão temporal sem embaralhamento.
- As transformações de escala foram colocadas dentro de `Pipeline`, portanto são ajustadas apenas com o conjunto de treino.

Os resultados são válidos para os arquivos desta avaliação e podem mudar caso as APIs sejam consultadas novamente e retornem dados diferentes.


# Entrega pelo GitHub

Envie **apenas o link do repositório público**. Organize-o com:

- `README.md` contendo objetivo, fontes e período dos dados, instruções para executar o notebook e resumo dos resultados;
- notebook `.ipynb` com a consulta às APIs, sua análise, os **três classificadores e os três regressores**, todos os imports necessários adicionados por você e conclusões em Markdown;
- os dois CSVs gerados (ou instruções completas e testadas para reproduzi-los) e eventuais figuras utilizadas.

Verifique se o notebook roda **na ordem das células** a partir de um ambiente limpo. Não inclua senhas, tokens ou arquivos sem relação com a avaliação. A banca deve conseguir encontrar as seis comparações e as respostas às duas tarefas sem procurar uma agulha no palheiro.

### Fontes

- [ANEEL — SIGA](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel)
- [ANEEL — recurso e campos usados](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel/resource/11ec447d-698d-4ab8-977f-b424d5deee6a)
- [Open-Meteo — histórico e unidades](https://open-meteo.com/en/docs/historical-weather-api)